In [27]:
import os
import json
import chromadb

from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown
from chromadb.config import Settings
from pathlib import Path

env_path = Path('.env')
load_dotenv(dotenv_path=env_path, override=True)




True

In [28]:
# ----------------- LLM & Clients Setup -----------------
llm_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.getenv("OpenRouterAPI"),
)

MODEL = "openai/gpt-oss-120b"
EMBEDDING_MODEL = "nvidia/nemotron-3.5-embed-8b:free"

In [29]:
# ----------------- Embedding Helper -----------------
def get_embedding(text: str):
    """Generates embeddings using OpenRouter API."""
    response = llm_client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )
    return response.data[0].embedding

In [ ]:
# ----------------- ChromaDB Setup (Persistent) -----------------
# Dedicated folder banayein taake database save rahe
DB_DIR = "./restaurant_vector_db"
chroma_client = chromadb.PersistentClient(path=DB_DIR)

collection = chroma_client.get_or_create_collection(
    name="restaurant_vector_db",
    metadata={"hnsw:space": "cosine"}
)

def init_knowledge_base(file_path="xander_restaurant_knowledge_base.md"):
    """Reads markdown file, chunks it, and indexes into ChromaDB."""
    if not os.path.exists(file_path):
        print(f"File '{file_path}' nahi mili! Creating sample file...")
        with open(file_path, "w", encoding="utf-8") as f:
            f.write("# Xander Restaurant Menu\n\n- Zinger Burger: Rs 850\n\n- Chicken Biryani: Rs 450\n\n- Club Sandwich: Rs 700\n")

    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Paragraph chunking
    raw_chunks = [chunk.strip() for chunk in content.split("\n\n") if chunk.strip()]
    
    current_count = collection.count()
    print(f"Current DB items before check: {current_count}")

    # Agar DB empty hai to naye chunks embed karke add karo
    if current_count == 0:
        print(f"Indexing {len(raw_chunks)} chunks into ChromaDB...")
        embeddings = [get_embedding(chunk) for chunk in raw_chunks]
        ids = [f"chunk_{i}" for i in range(len(raw_chunks))]
        
        collection.add(
            documents=raw_chunks,
            embeddings=embeddings,
            ids=ids
        )
        print(f"Done! Successfully stored {collection.count()} chunks.")
    else:
        print(f"Collection already has {current_count} chunks. Skipping re-indexing.")

# Run initialization
    init_knowledge_base()

In [51]:



# ----------------- Tools Implementation -----------------
complaint_list = []
OrderId = [
    "OR-1174",
    "OR-1004",
    "OR-2074",
    "OR-1014"
]

In [52]:
def search_knowledge_base(query: str):
    """Retrieves relevant restaurant facts and prints chunk count and text."""
    query_emb = get_embedding(query)
    
    # n_results control karta hai ke maximum kitne chunks retrieve karne hain
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=3,  # Aap jitne chunks dekhna chahein yahan change kar sakte hain
        include=["documents", "distances"]
    )
    
    docs = results.get("documents", [[]])[0]
    distances = results.get("distances", [[]])[0] if results.get("distances") else []
    ids = results.get("ids", [[]])[0] if results.get("ids") else []
    
    # Console par chunk count aur preview print karna
    print("\n" + "=" * 20 + " [CHROMA CHUNK RETRIEVAL] " + "=" * 20)
    print(f"Search Query: '{query}'")
    print(f"Total Chunks Retrieved: {len(docs)}")
    
    for idx, doc in enumerate(docs):
        dist_str = f" | Distance: {distances[idx]:.4f}" if idx < len(distances) else ""
        chunk_id = f"[{ids[idx]}]" if idx < len(ids) else ""
        print(f"\n--- Chunk {idx + 1} {chunk_id}{dist_str} ---")
        print(doc)
    print("=" * 66 + "\n")
    
    return {
        "total_chunks_found": len(docs),
        "knowledge_snippets": docs
    }

def log_complaint(customer_name, order_id, phone, complaint):
    temp_dict = {
        "customer_name": customer_name,
        "order_id": order_id,
        "phone": phone,
        "complaint": complaint,
        "status": "Pending"
    }
    complaint_list.append(temp_dict)
    return {
        "message": f"Saved! Total Complaints: {len(complaint_list)}"
    }

def Validate_Order_id(orderid):
    if orderid in OrderId:
        return {
            "valid": True,
            "message": "Order ID is valid."
        }
    else:
        return {
            "valid": False,
            "message": "Invalid Order ID. Please provide a correct order ID."
        }

In [53]:
# ----------------- Tool Definitions for OpenAI Schema -----------------
tools = [
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": "Search restaurant knowledge base for menu items, pricing, timings, ingredients, and policies.",
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Search term or question regarding menu, food, or restaurant policies."
                    }
                },
                "required": ["query"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "log_complaint",
            "description": (
                "Log a complaint ONLY after Validate_Order_id returns valid=True. "
                "NEVER call this tool if order ID validation returns valid=False."
            ),
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "customer_name": {
                        "type": "string",
                        "description": "Customer name logging the complaint."
                    },
                    "order_id": {
                        "type": "string",
                        "description": "The validated order ID (e.g., OR-0000)."
                    },
                    "phone": {
                        "type": "string",
                        "description": "Customer phone number or '-'."
                    },
                    "complaint": {
                        "type": "string",
                        "description": "Customer complaint summary (max 8 words)."
                    }
                },
                "required": ["customer_name", "order_id", "phone", "complaint"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "Validate_Order_id",
            "description": (
                "Validate the customer's order ID against the OrderId list. "
                "This tool MUST be called before log_complaint. "
                "If valid=False, stop and ask the customer for a correct ID."
            ),
            "strict": True,
            "parameters": {
                "type": "object",
                "properties": {
                    "orderid": {
                        "type": "string",
                        "description": "The customer's order ID, for example OR-1174."
                    }
                },
                "required": ["orderid"],
                "additionalProperties": False
            }
        }
    }
]

In [54]:
# ----------------- System Prompt -----------------
messages = [
    {
        "role": "system",
        "content": """
You are the AI Customer Support Assistant for Xander Restaurant.

### Tools:
1. `search_knowledge_base`: Retrieve details regarding menu, ingredients, deals, and restaurant policies. Call this when answering food/menu questions.
2. `Validate_Order_id`: Call before logging any complaint to verify if the order ID exists.
3. `log_complaint`: Call ONLY after `Validate_Order_id` returns valid=True.

### Complaint Workflow:
1. Call `Validate_Order_id` first.
2. If valid=False, politely inform the customer that the order ID does not exist in our system and ask for the correct order ID. STOP there. Do not call `log_complaint`.
3. If valid=True, proceed to log the complaint via `log_complaint`.

### Guidelines:
- Keep replies brief, professional, and empathetic.
- Prices must be displayed in Pakistani Rupees (e.g., "Rs 850").
"""
    }
]

function_names = {
    "search_knowledge_base": search_knowledge_base,
    "log_complaint": log_complaint,
    "Validate_Order_id": Validate_Order_id
}

In [55]:
# ----------------- Execution Loop -----------------
def ask_assistant(question):
    messages.append({
        "role": "user",
        "content": question
    })

    while True:
        completion = llm_client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )

        reply = completion.choices[0].message

        if not reply.tool_calls:
            messages.append({"role": "assistant", "content": reply.content})
            return Markdown(reply.content or "")

        messages.append(reply)

        for call in reply.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            function_name = call.function.name

            result = function_names[function_name](**args)
            print("Tool called:", function_name, args, result)

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result)
            })

In [56]:
# Example run:
ask_assistant("I want to know the menu")


==================== [CHROMA CHUNK RETRIEVAL] ====================
Search Query: 'menu'
Total Chunks Retrieved: 0

Tool called: search_knowledge_base {'query': 'menu'} {'total_chunks_found': 0, 'knowledge_snippets': []}


Here’s a brief overview of our menu categories:

- **Starters & Soups**
- **Main Courses** (including grilled, biryani, and chef’s specials)
- **Vegetarian & Healthy Options**
- **Desserts & Sweet Treats**
- **Beverages & Fresh Juices**

If you’d like details on a specific dish, its ingredients, or pricing (e.g., “Rs 850”), just let me know!

In [57]:
print("Total Chunks in DB:", collection.count())
print("Sample Data:", collection.peek(limit=2))

Total Chunks in DB: 0
Sample Data: {'ids': [], 'embeddings': array([], dtype=float64), 'documents': [], 'uris': None, 'included': ['metadatas', 'documents', 'embeddings'], 'data': None, 'metadatas': []}


### Stop

In [6]:
chroma_client = chromadb.PersistentClient(path="./")

collection = chroma_client.get_or_create_collection(name="restaurant_vector_db")

In [ ]:
# llm_client = OpenAI(
#     base_url="https://openrouter.ai/api/v1",
#     api_key=os.getenv("OPEN_ROUTER_KEY"),
# )

llm_client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY"),
)

# MODEL = "nvidia/nemotron-3.5-lightning:free"       # chat model (free, NVIDIA)
MODEL = "openai/gpt-oss-120b"
EMBEDDING_MODEL = "nvidia/nemotron-3.5-embed-8b:free"

In [ ]:
def create_embeddings(texts):
    r = llm_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)    # one request for the whole list
    return [item.embedding for item in r.data]     

## Load + chunk

The same code as Day 30. The documents are in `chai_wala_docs`.

In [ ]:
from pathlib import Path

def load_documents(folder):
    docs = []
    for path in sorted(Path(folder).glob("*.txt")):              # every .txt file in the folder
        docs.append({"source": path.name, "text": path.read_text(encoding="utf-8")})
    return docs

def chunk_by_paragraph(text):
    parts = [p.strip() for p in text.split("\n\n")]              # a blank line = a new paragraph
    return [p for p in parts if len(p) > 30]                      # skip very short parts, like titles

def chunk_documents(docs):
    chunks = []
    for doc in docs:
        for i, text in enumerate(chunk_by_paragraph(doc["text"])):
            chunks.append({"id": f"{doc['source']}-{i}", "source": doc["source"], "text": text})
    return chunks

chunks = chunk_documents(load_documents("chai_wala_docs"))
print(len(chunks), "chunks")

In [ ]:
menu = [
    {"name": "Zinger Burger", "price": 850},
    {"name": "Chicken Biryani", "price": 450},
    {"name": "Club Sandwich", "price": 700}
]

complaint_list = []

order_list = [
    "OR-1045",
    "OR-1046",
    "OR-1047"
]

In [ ]:
def fetch_menu():
    return menu

def log_complaint(customer_name, order_id, phone, complaint):
    temp_dict = {
        "customer_name": customer_name,
        "order_id": order_id,
        "phone": phone,
        "complaint": complaint,
        "status": "Pending"
    }
    complaint_list.append(temp_dict)

    return {
        "message": f"Saved! Total Complaint: {len(complaint_list)}"
    }

def validate_order_id(order_id):
    if order_id in order_list:
        return True
    return False


tools = [
    {
        "type": "function",
        "function": {
                "name": "fetch_menu",
                "description": "Get the restaurant menu",
                # no inputs
                "parameters": {"type": "object", "properties": {}},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "validate_order_id",
            "description": "Validate order id before calling log_complaint tool it return True/False",
            "parameters": {
                "type": "object", 
                "properties": {
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                },
                "required": ["order_id"],
                "additionalProperties": False,

            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "log_complaint",
            "description": "Log the complain",
            "strict": True,
            "parameters": {
                "type": "object", 
                "properties": {
                    "customer_name": {
                        "type": "string",
                        "description": "Customer name who is logging complain.",
                    },
                    "order_id": {
                        "type": "string",
                        "description": "the order id of which customer ordered food. OrderId should be like this OR-0000",
                    },
                    "phone": {
                        "type": "string",
                        "description": "Customer phone number or '-' ",
                    },
                    "complaint": {
                        "type": "string",
                        "description": "customer complain should me not more than 8 words",
                    }
                },
                "required": ["customer_name", "order_id", "phone", "complaint"],
                "additionalProperties": False,

            },
        }
    }

]

In [ ]:
messages = [
    {
      "role": "system",  # User | Assistant | System
      "content": """
       You are the AI Customer Support Assistant for **Xander Restaurant**. Your job is to answer customer queries in a friendly, professional, and helpful way.

        ### Tools
        1. **`fetch_menu`** — Call whenever customers ask about menu items, prices, ingredients, or food recommendations. Use the tool's response to provide accurate information.
        2. **`log_complaint`** — Call whenever a customer raises a complaint. Collect the necessary details and log the complaint using this schema:

        ```python
        complaint = {
            "customer_name": "",
            "phone": "",
            "complaint": "",
            "order_id": "",
            "status": "Pending"
        }
        ```

        3. **`validate_order_id`** — Before logging complaint trigger this tool call validate_order_id to validate wether the order id is valid or not then call log_complaint


        Ask for missing necessary details before logging. The `order_id` can be empty if unavailable. Set the initial status to `"Pending"`. Confirm the complaint is logged only after the tool succeeds.

        ### Guidelines
        - Keep replies short, natural, and friendly.
        - Never invent menu details, prices, or restaurant policies.
        - If you don't know something, politely say so.
        - Make every customer feel valued.
        - All prices are in Pakistani Rupees. Write them as "Rs 850".
      """
    }
]


In [ ]:
function_names = {
    "fetch_menu": fetch_menu,
    "log_complaint": log_complaint,
    "validate_order_id": validate_order_id
}

def ask_assistant(question):

    messages.append({
        "role": "user", 
        "content": question
    })

    while True:
        completion = llm_client.chat.completions.create(
                model=MODEL, 
                messages=messages, 
                tools=tools,
                temperature=0
            )
           
        reply = completion.choices[0].message
        messages.append(reply)
    
        # reply_json = reply.model_dump(exclude_none=True)
        # reply_json.pop("reasoning", None)            # hide the model's long thinking
        # reply_json.pop("reasoning_details", None)
        # print("LLM REPLY:", json.dumps(reply_json, indent=2))
    
        if not reply.tool_calls: 
            return Markdown(reply.content)
            break                        # model asked for a tool
       
        for call in reply.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            result = function_names[call.function.name](**args)     # run our Python function
            print("Tool called:", call.function.name, args, result)
            messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})        

In [ ]:
ask_assistant("i ordered burger but roll was served WTH!!! and the bill was made with burger, my name is Sam, OR-1045, phone is 0322342423")

In [ ]:
print(json.dumps(complaint_list))



Create a new tool validate_order to validate orderId when logging complaint
if order id does not gets validated say user please provider correct Order ID